In [ ]:
# stdlib pathlib (filesystem paths)
from pathlib import Path

# src/utils/pathing.py
from src.utils.pathing import ensure_repo_root_on_sys_path  # src/utils/pathing.py

ensure_repo_root_on_sys_path(Path.cwd())

# 27. Unsupervised Learning: Gaussian Mixture Models (GMM)

## Algorithm Category
**Type**: Unsupervised Learning - Clustering & Density Estimation  
**Complexity**: Medium-High  
**Use Case**: Probabilistic clustering using mixture of Gaussian distributions

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand Gaussian Mixture Models and probabilistic clustering
- Implement GMM using Expectation-Maximization (EM) algorithm
- Understand soft clustering vs hard clustering
- Use information criteria (AIC, BIC) to select number of components
- Visualize probability distributions and cluster assignments
- Apply GMM to real-world problems

## Historical Context

GMM with EM algorithm was popularized in the 1970s-1980s:
- Dempster, A.P., et al. (1977): "Maximum likelihood from incomplete data via the EM algorithm"
- One of the most important algorithms in statistics and machine learning
- Foundation for many probabilistic models

**Key Papers/References:**
- Dempster, A.P., et al. (1977). "Maximum likelihood from incomplete data via the EM algorithm"
- McLachlan, G. & Peel, D. (2000). "Finite Mixture Models"

## When to Use Gaussian Mixture Models

GMM is appropriate when:
- You need soft/probabilistic clustering
- Data can be modeled as mixture of Gaussians
- You want to estimate probability distributions
- Clusters may overlap
- You need uncertainty estimates for cluster assignments
- Working with continuous data

## Theory & Mechanics

### Mathematical Foundation

GMM models data as a mixture of K Gaussian distributions:

**Probability Density Function:**
$$p(x) = \sum_{k=1}^{K} \pi_k \mathcal{N}(x | \mu_k, \Sigma_k)$$

Where:
- $\pi_k$: Mixing coefficient (weight) for component $k$, $\sum_{k=1}^{K} \pi_k = 1$
- $\mu_k$: Mean of component $k$
- $\Sigma_k$: Covariance matrix of component $k$
- $\mathcal{N}(x | \mu_k, \Sigma_k)$: Multivariate Gaussian distribution

**Expectation-Maximization (EM) Algorithm:**

**E-Step (Expectation):**
$$\gamma_{ik} = \frac{\pi_k \mathcal{N}(x_i | \mu_k, \Sigma_k)}{\sum_{j=1}^{K} \pi_j \mathcal{N}(x_i | \mu_j, \Sigma_j)}$$

**M-Step (Maximization):**
$$\mu_k = \frac{\sum_{i=1}^{N} \gamma_{ik} x_i}{\sum_{i=1}^{N} \gamma_{ik}}$$

$$\Sigma_k = \frac{\sum_{i=1}^{N} \gamma_{ik} (x_i - \mu_k)(x_i - \mu_k)^T}{\sum_{i=1}^{N} \gamma_{ik}}$$

$$\pi_k = \frac{1}{N} \sum_{i=1}^{N} \gamma_{ik}$$

### How It Works

1. **Initialize**: Randomly initialize parameters ($\mu_k$, $\Sigma_k$, $\pi_k$)
2. **E-Step**: Calculate responsibility (posterior probability) of each point belonging to each component
3. **M-Step**: Update parameters using weighted maximum likelihood
4. **Repeat**: Steps 2-3 until convergence

### Key Hyperparameters

- **n_components**: Number of mixture components (clusters)
- **covariance_type**: Type of covariance matrix
  - 'full': Each component has its own general covariance matrix
  - 'tied': All components share the same covariance matrix
  - 'diag': Each component has its own diagonal covariance matrix
  - 'spherical': Each component has its own single variance
- **max_iter**: Maximum iterations for EM algorithm
- **tol**: Convergence threshold

### Advantages

- Provides soft clustering (probabilistic assignments)
- Can model overlapping clusters
- Handles elliptical clusters (not just spherical)
- Estimates probability distributions
- Uses information criteria (AIC, BIC) for model selection

### Limitations

- Assumes Gaussian distribution (may not fit all data)
- Sensitive to initialization (local optima)
- Can be slow for large datasets
- Requires specifying number of components
- May struggle with non-Gaussian clusters


## Implementation

Let's implement Gaussian Mixture Models.


In [ ]:
# ============================================
# IMPORTING LIBRARIES: Setting Up Our Tools
# ============================================

# Core data science libraries
import numpy as np  # NumPy: Numerical computing (arrays, math operations)
import pandas as pd  # Pandas: Data manipulation (DataFrames, data analysis)
import matplotlib.pyplot as plt  # Matplotlib: Plotting and visualization

# Scikit-learn: Machine learning library
from sklearn.datasets import (
    make_blobs,  # Generate synthetic blob-shaped clusters (for testing)
    load_iris  # Iris flower dataset (real-world example)
)
from sklearn.mixture import GaussianMixture  # Gaussian Mixture Model (probabilistic clustering)
from sklearn.preprocessing import StandardScaler  # Feature scaling (normalization)
from sklearn.metrics import silhouette_score  # Calculate silhouette score (cluster quality metric)

# ============================================
# IMPORTING OUR HELPER FUNCTIONS
# ============================================

# Our custom utility functions (organized in src/ directory)
from src.models.unsupervised import evaluate_clustering  # Evaluate clustering quality

print("Libraries imported successfully!")  # Confirm all imports worked


In [ ]:
# ============================================
# GENERATING SYNTHETIC DATASET: Testing GMM
# ============================================

# make_blobs() generates synthetic data with spherical clusters
# This is perfect for testing GMM, which can model Gaussian distributions
# GMM can handle elliptical clusters (not just spherical like K-Means)

# make_blobs() parameters:
# n_samples=300: Number of data points to generate
# centers=3: Number of clusters (we know there are 3)
# n_features=2: Number of features (2D for easy visualization)
# random_state=42: Ensures reproducible results
# cluster_std=0.60: Standard deviation of clusters (controls spread)
X, y_true = make_blobs(n_samples=300, centers=3, n_features=2, 
                       random_state=42, cluster_std=0.60)
# Returns:
# - X: Feature values (300 samples × 2 features)
# - y_true: True cluster labels (for comparison - we won't use these for clustering!)

print(f"Dataset Shape: {X.shape}")  # Output: (300, 2) - 300 points, 2 features
print(f"True number of clusters: {len(np.unique(y_true))}")  # Output: 3 clusters

# ============================================
# APPLYING GAUSSIAN MIXTURE MODEL
# ============================================

# GaussianMixture models data as a mixture of K Gaussian distributions
# Unlike K-Means (hard clustering), GMM provides soft clustering (probabilities)
# Uses Expectation-Maximization (EM) algorithm to estimate parameters

# GaussianMixture parameters:
# n_components=3: Number of Gaussian components (clusters)
#   - We know there are 3 clusters from the data generation
#   - In practice, you'd use AIC/BIC to select this
#
# random_state=42: Ensures reproducible results
#   - EM algorithm starts with random initialization
#   - Same seed = same starting point = same results
#
# covariance_type='full': Type of covariance matrix
#   - 'full': Each component has its own general covariance matrix (most flexible)
#   - 'tied': All components share the same covariance matrix
#   - 'diag': Each component has its own diagonal covariance matrix
#   - 'spherical': Each component has its own single variance (like K-Means)
gmm = GaussianMixture(n_components=3, random_state=42, covariance_type='full')

# ============================================
# MODEL TRAINING: Expectation-Maximization
# ============================================

# .fit() trains the GMM using EM algorithm:
# E-Step: Calculate responsibility (probability) of each point belonging to each component
# M-Step: Update parameters (means, covariances, mixing weights) using weighted MLE
# Repeats until convergence (parameters stop changing)
gmm.fit(X)  # Train the model

# ============================================
# MAKING PREDICTIONS: Hard vs Soft Clustering
# ============================================

# .predict() returns hard cluster assignments (most likely component)
# This is similar to K-Means - each point assigned to one cluster
y_pred = gmm.predict(X)  # Hard clustering: array of cluster labels (0, 1, or 2)

# .predict_proba() returns soft cluster assignments (probabilities)
# Each point gets a probability distribution over all components
# This is unique to GMM - K-Means doesn't provide probabilities!
y_proba = gmm.predict_proba(X)
# Returns: array of shape (n_samples, n_components)
# Each row sums to 1.0 (probabilities)
# Example: [0.1, 0.8, 0.1] means 10% chance cluster 0, 80% chance cluster 1, 10% chance cluster 2

# ============================================
# DISPLAYING MODEL INFORMATION
# ============================================

print(f"\nGMM Results:")
print(f"  Number of components: {gmm.n_components}")  # Number of Gaussian components (3)
print(f"  Covariance type: {gmm.covariance_type}")  # Type of covariance ('full')
print(f"  Converged: {gmm.converged_}")  # True if EM algorithm converged
print(f"  Number of iterations: {gmm.n_iter_}")  # How many EM iterations were needed

# ============================================
# VISUALIZING CLUSTERING RESULTS
# ============================================

# Create figure with 3 subplots
plt.figure(figsize=(15, 5))  # Width=15 inches, height=5 inches

# Subplot 1: True clusters (ground truth)
plt.subplot(1, 3, 1)  # 1 row, 3 columns, position 1 (left)

# Scatter plot colored by true labels
plt.scatter(X[:, 0], X[:, 1], c=y_true, cmap='viridis', s=50, alpha=0.7)
# c=y_true: Color by true cluster labels
plt.title('True Clusters')  # Chart title
plt.xlabel('Feature 1')  # X-axis label
plt.ylabel('Feature 2')  # Y-axis label
plt.grid(True, alpha=0.3)  # Add grid

# Subplot 2: Hard clustering (GMM predictions)
plt.subplot(1, 3, 2)  # 1 row, 3 columns, position 2 (middle)

# Scatter plot colored by hard cluster assignments
plt.scatter(X[:, 0], X[:, 1], c=y_pred, cmap='viridis', s=50, alpha=0.7)
# c=y_pred: Color by predicted cluster (hard assignment)

# Plot component means (centers of Gaussian distributions)
plt.scatter(gmm.means_[:, 0], gmm.means_[:, 1], c='red', marker='x', 
           s=200, linewidths=3, label='Means')
# gmm.means_: Mean (center) of each Gaussian component
# Red X marks show where each Gaussian is centered

plt.title('GMM Hard Clustering')  # Chart title
plt.xlabel('Feature 1')  # X-axis label
plt.ylabel('Feature 2')  # Y-axis label
plt.legend()  # Show legend
plt.grid(True, alpha=0.3)  # Add grid

# Subplot 3: Soft clustering (probabilities)
plt.subplot(1, 3, 3)  # 1 row, 3 columns, position 3 (right)

# Color by probability of belonging to cluster 0
plt.scatter(X[:, 0], X[:, 1], c=y_proba[:, 0], cmap='Reds', s=50, alpha=0.7)
# c=y_proba[:, 0]: Color by probability of cluster 0
# y_proba[:, 0] extracts probability of cluster 0 for each point
# Red colormap: Darker red = higher probability, lighter = lower probability

# Plot component means
plt.scatter(gmm.means_[:, 0], gmm.means_[:, 1], c='blue', marker='x', 
           s=200, linewidths=3, label='Means')
# Blue X marks show component centers

plt.title('GMM Soft Clustering\n(Probability of Cluster 0)')  # Chart title
plt.xlabel('Feature 1')  # X-axis label
plt.ylabel('Feature 2')  # Y-axis label
plt.colorbar(label='Probability')  # Color legend showing probability scale
plt.legend()  # Show legend
plt.grid(True, alpha=0.3)  # Add grid

# Adjust layout
plt.tight_layout()
plt.show()  # Display all plots

# Interpretation:
# - Left plot: True clusters (what we're trying to find)
# - Middle plot: Hard clustering (each point assigned to one cluster, like K-Means)
# - Right plot: Soft clustering (probabilities show uncertainty - unique to GMM!)
# - Points near boundaries have lower probabilities (uncertainty)
# - Points far from boundaries have high probabilities (certainty)
# - GMM means (red/blue X's) are centers of Gaussian distributions


## Model Selection: AIC and BIC

Let's use information criteria to select the optimal number of components.


In [ ]:
# ============================================
# MODEL SELECTION: Finding Optimal Number of Components
# ============================================

# Unlike K-Means, GMM provides information criteria (AIC, BIC) for model selection
# These criteria balance model fit (likelihood) with complexity (number of parameters)
# Lower AIC/BIC = better model (better fit with less complexity)

# Test different numbers of components
n_components_range = range(1, 11)  # Test k from 1 to 10
aic_scores = []  # Store AIC scores
bic_scores = []  # Store BIC scores
silhouette_scores = []  # Store silhouette scores (for comparison)

# Test each number of components
for n_comp in n_components_range:
    # Create GMM with this number of components
    gmm_test = GaussianMixture(n_components=n_comp, random_state=42, covariance_type='full')
    
    # Train the model
    gmm_test.fit(X)  # Train on data
    
    # Calculate AIC (Akaike Information Criterion)
    # AIC = -2 * log_likelihood + 2 * n_parameters
    # Lower is better (penalizes complexity less than BIC)
    aic_scores.append(gmm_test.aic(X))  # AIC score for this model
    
    # Calculate BIC (Bayesian Information Criterion)
    # BIC = -2 * log_likelihood + log(n_samples) * n_parameters
    # Lower is better (penalizes complexity more than AIC)
    bic_scores.append(gmm_test.bic(X))  # BIC score for this model
    
    # Calculate silhouette score (requires at least 2 clusters)
    if n_comp > 1:
        # Get hard cluster assignments
        labels = gmm_test.predict(X)  # Cluster labels
        
        # Calculate silhouette score (cluster quality metric)
        sil_score = silhouette_score(X, labels)
        # Range: -1 to 1, higher is better
        silhouette_scores.append(sil_score)
    else:
        # Can't calculate silhouette with 1 cluster
        silhouette_scores.append(-1)  # Placeholder value

# ============================================
# VISUALIZING INFORMATION CRITERIA
# ============================================

# Create figure with 2 subplots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))  # 1 row, 2 columns

# Plot 1: AIC and BIC
axes[0].plot(n_components_range, aic_scores, 'o-', label='AIC', markersize=6)
axes[0].plot(n_components_range, bic_scores, 's-', label='BIC', markersize=6)
# X-axis: number of components, Y-axis: AIC/BIC score
# Lower scores = better models

# Draw vertical line at true k
axes[0].axvline(x=3, color='r', linestyle='--', label='True k=3')
# Red dashed line shows true number of clusters

axes[0].set_xlabel('Number of Components')  # X-axis label
axes[0].set_ylabel('Score')  # Y-axis label (lower is better)
axes[0].set_title('AIC and BIC for Model Selection')  # Chart title
axes[0].legend()  # Show legend
axes[0].grid(True, alpha=0.3)  # Add grid

# Plot 2: Silhouette Score
axes[1].plot(n_components_range, silhouette_scores, '^-', color='green', markersize=6)
# X-axis: number of components, Y-axis: silhouette score
# Higher scores = better models

# Draw vertical line at true k
axes[1].axvline(x=3, color='r', linestyle='--', label='True k=3')
# Red dashed line shows true number of clusters

axes[1].set_xlabel('Number of Components')  # X-axis label
axes[1].set_ylabel('Silhouette Score')  # Y-axis label (higher is better)
axes[1].set_title('Silhouette Score')  # Chart title
axes[1].grid(True, alpha=0.3)  # Add grid
axes[1].legend()  # Show legend

# Adjust layout
plt.tight_layout()
plt.show()  # Display both plots

# ============================================
# FINDING OPTIMAL NUMBER OF COMPONENTS
# ============================================

# Find k with minimum AIC (best model according to AIC)
optimal_aic = n_components_range[np.argmin(aic_scores)]
# np.argmin(): Index of minimum value
# n_components_range[...]: Get k value at that index

# Find k with minimum BIC (best model according to BIC)
optimal_bic = n_components_range[np.argmin(bic_scores)]
# np.argmin(): Index of minimum value

# Find k with maximum silhouette score (best model according to silhouette)
optimal_sil = n_components_range[np.argmax(silhouette_scores)]
# np.argmax(): Index of maximum value

print(f"Optimal number of components:")
print(f"  AIC: {optimal_aic}")  # Best k according to AIC
print(f"  BIC: {optimal_bic}")  # Best k according to BIC
print(f"  Silhouette: {optimal_sil}")  # Best k according to silhouette
print(f"  True: 3")  # True number of clusters

# Interpretation:
# - AIC and BIC should have minimum at true k (or close to it)
# - BIC usually penalizes complexity more (prefers simpler models)
# - Silhouette score should peak at true k
# - If criteria disagree, BIC is often more conservative (prefers fewer components)
# - In practice, use AIC/BIC for GMM (they're designed for probabilistic models)


## Covariance Types

Let's compare different covariance types.


In [ ]:
# ============================================
# COMPARING COVARIANCE TYPES: Flexibility vs Complexity
# ============================================

# Different covariance types control the shape of Gaussian distributions
# More flexible = better fit but more parameters (risk of overfitting)
# Less flexible = simpler model but may not fit data well

# Test different covariance types
covariance_types = ['full', 'tied', 'diag', 'spherical']
# - 'full': Each component has its own general covariance matrix (most flexible)
# - 'tied': All components share the same covariance matrix
# - 'diag': Each component has its own diagonal covariance (no correlations)
# - 'spherical': Each component has its own single variance (like K-Means)

# Create figure with 4 subplots (one per covariance type)
fig, axes = plt.subplots(1, 4, figsize=(16, 4))  # 1 row, 4 columns

# Test each covariance type
for idx, cov_type in enumerate(covariance_types):
    # Create GMM with this covariance type
    gmm_cov = GaussianMixture(n_components=3, random_state=42, covariance_type=cov_type)
    
    # Train the model
    gmm_cov.fit(X)  # Train on data
    
    # Get hard cluster assignments
    labels_cov = gmm_cov.predict(X)  # Cluster labels
    
    # Plot clusters
    axes[idx].scatter(X[:, 0], X[:, 1], c=labels_cov, cmap='viridis', s=50, alpha=0.7)
    # c=labels_cov: Color by cluster assignment
    
    # Plot component means
    axes[idx].scatter(gmm_cov.means_[:, 0], gmm_cov.means_[:, 1], c='red', marker='x', 
                     s=200, linewidths=3)
    # Red X marks show component centers
    
    # Title shows covariance type and AIC score
    axes[idx].set_title(f'{cov_type.capitalize()}\nAIC: {gmm_cov.aic(X):.1f}')
    # Lower AIC = better model (for this covariance type)
    
    axes[idx].set_xlabel('Feature 1')  # X-axis label
    axes[idx].set_ylabel('Feature 2')  # Y-axis label
    axes[idx].grid(True, alpha=0.3)  # Add grid

# Adjust layout
plt.tight_layout()
plt.show()  # Display all plots

# ============================================
# COMPARING COVARIANCE TYPES: Quantitative Comparison
# ============================================

print("\nComparison of covariance types:")
for cov_type in covariance_types:
    # Create and train GMM with this covariance type
    gmm_cov = GaussianMixture(n_components=3, random_state=42, covariance_type=cov_type)
    gmm_cov.fit(X)  # Train on data
    
    # Display AIC and BIC scores
    print(f"  {cov_type}: AIC = {gmm_cov.aic(X):.1f}, BIC = {gmm_cov.bic(X):.1f}")
    # Lower scores = better model
    # Compare across covariance types to find best one

# Interpretation:
# - 'full': Most flexible, can model elliptical clusters of any orientation
# - 'tied': All clusters have same shape (less flexible, fewer parameters)
# - 'diag': Clusters are axis-aligned ellipses (no rotation)
# - 'spherical': Clusters are circles (like K-Means, least flexible)
# - For blob data, 'full' usually works best (can model any ellipse)
# - For simpler data, 'spherical' or 'diag' may be sufficient (fewer parameters)
# - Choose covariance type with lowest AIC/BIC (best fit with reasonable complexity)


## Validation & Testing

Let's validate the model and compare with K-Means.


In [ ]:
# ============================================
# VALIDATION 1: Evaluating Clustering Quality
# ============================================

# Evaluate clustering quality using silhouette score
# evaluate_clustering() calculates quality metrics
evaluation = evaluate_clustering(X, y_pred, algorithm='GMM')
# X: Feature data
# y_pred: Cluster assignments
# algorithm='GMM': Specify algorithm (for reporting)
# Returns dictionary with evaluation metrics

print("Clustering Evaluation:")
print(f"  Silhouette Score: {evaluation['silhouette_score']:.3f}")  # Quality metric (-1 to 1)
print(f"  Number of clusters: {evaluation['n_clusters']}")  # Number of clusters found

# ============================================
# VALIDATION 2: Comparing GMM with K-Means
# ============================================

# GMM and K-Means are both clustering algorithms but work differently
# GMM: Probabilistic, soft clustering, can model elliptical clusters
# K-Means: Deterministic, hard clustering, assumes spherical clusters
# We'll compare them on the same data to see the differences

from sklearn.cluster import KMeans  # K-Means clustering

# Create K-Means with 3 clusters (same as GMM)
kmeans = KMeans(n_clusters=3, random_state=42)
# n_clusters=3: Force 3 clusters (K-Means requires specifying k)

# Get K-Means cluster assignments
y_kmeans = kmeans.fit_predict(X)  # Cluster labels (0, 1, or 2)

# ============================================
# VISUALIZING COMPARISON: GMM vs K-Means
# ============================================

# Create figure with 2 subplots
plt.figure(figsize=(12, 5))  # Width=12 inches, height=5 inches

# Subplot 1: GMM Results
plt.subplot(1, 2, 1)  # 1 row, 2 columns, position 1 (left)

# Scatter plot colored by GMM clusters
plt.scatter(X[:, 0], X[:, 1], c=y_pred, cmap='viridis', s=50, alpha=0.7)
# c=y_pred: Color by GMM cluster assignment

# Plot GMM component means
plt.scatter(gmm.means_[:, 0], gmm.means_[:, 1], c='red', marker='x', 
           s=200, linewidths=3, label='GMM Means')
# Red X marks show centers of Gaussian distributions
# gmm.means_: Mean (center) of each Gaussian component

plt.title('GMM (Soft Clustering)')  # Chart title
plt.xlabel('Feature 1')  # X-axis label
plt.ylabel('Feature 2')  # Y-axis label
plt.legend()  # Show legend
plt.grid(True, alpha=0.3)  # Add grid

# Subplot 2: K-Means Results
plt.subplot(1, 2, 2)  # 1 row, 2 columns, position 2 (right)

# Scatter plot colored by K-Means clusters
plt.scatter(X[:, 0], X[:, 1], c=y_kmeans, cmap='viridis', s=50, alpha=0.7)
# c=y_kmeans: Color by K-Means cluster assignment

# Plot K-Means cluster centers (centroids)
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
           c='red', marker='x', s=200, linewidths=3, label='K-Means Centroids')
# Red X marks show cluster centers
# kmeans.cluster_centers_: Coordinates of centroids

plt.title('K-Means (Hard Clustering)')  # Chart title
plt.xlabel('Feature 1')  # X-axis label
plt.ylabel('Feature 2')  # Y-axis label
plt.legend()  # Show legend
plt.grid(True, alpha=0.3)  # Add grid

# Adjust layout
plt.tight_layout()
plt.show()  # Display both plots

# ============================================
# COMPARISON SUMMARY
# ============================================

print("\nComparison:")
print(f"  GMM AIC: {gmm.aic(X):.1f}")  # AIC score (lower is better)
print(f"  GMM BIC: {gmm.bic(X):.1f}")  # BIC score (lower is better)
print(f"  GMM provides probability estimates, K-Means does not")
# GMM can tell you "80% chance cluster 1, 20% chance cluster 2"
# K-Means only says "cluster 1" (no uncertainty)

# ============================================
# ASSERTIONS: Automated Validation Checks
# ============================================

# Check 1: GMM should converge
assert gmm.converged_, "GMM should converge"
# If not converged, EM algorithm didn't finish (may need more iterations)

# Check 2: Silhouette score should be positive
assert evaluation['silhouette_score'] > 0, "Silhouette score should be positive"
# Negative silhouette = points assigned to wrong clusters

print("\n✓ Validation checks passed")  # All checks passed!

# Interpretation:
# - GMM: Probabilistic model, provides uncertainty estimates
# - K-Means: Deterministic model, no uncertainty
# - For spherical clusters, both work similarly
# - For elliptical clusters, GMM with 'full' covariance is better
# - GMM can model overlapping clusters (probabilities), K-Means cannot
# - GMM provides AIC/BIC for model selection, K-Means does not


## Real-World Application

Let's apply GMM to the Iris dataset.


In [ ]:
# ============================================
# REAL-WORLD APPLICATION: Iris Dataset
# ============================================

# Load Iris dataset (real-world classification problem)
iris = load_iris()  # Returns Bunch object
X_iris = iris.data  # Features: flower measurements (150 samples × 4 features)
y_iris = iris.target  # True labels: flower species (for comparison)

# ============================================
# FEATURE SCALING: Important for GMM
# ============================================

# GMM is sensitive to feature scaling (unlike K-Means, which uses distances)
# Features on different scales can distort the Gaussian distributions
# Standardize features to have mean=0 and std=1

scaler = StandardScaler()  # StandardScaler normalizes features
X_iris_scaled = scaler.fit_transform(X_iris)
# fit_transform(): Learn scaling from data and apply it
# X_iris_scaled: Features normalized (mean=0, std=1 for each column)

# ============================================
# APPLYING GMM TO IRIS DATASET
# ============================================

# Create GMM with 3 components (we know there are 3 species)
gmm_iris = GaussianMixture(n_components=3, random_state=42, covariance_type='full')
# n_components=3: 3 clusters (setosa, versicolor, virginica)
# covariance_type='full': Most flexible (can model elliptical clusters)

# Train the model
gmm_iris.fit(X_iris_scaled)  # Train on scaled data

# Get hard cluster assignments
y_iris_pred = gmm_iris.predict(X_iris_scaled)  # Hard clustering: cluster labels

# Get soft cluster assignments (probabilities)
y_iris_proba = gmm_iris.predict_proba(X_iris_scaled)
# Returns: array of shape (150, 3) - probability distribution for each sample

# ============================================
# EVALUATING CLUSTERING PERFORMANCE
# ============================================

# Calculate silhouette score (cluster quality metric)
sil_score_iris = silhouette_score(X_iris_scaled, y_iris_pred)
# Range: -1 to 1, higher is better

print("Iris Dataset Clustering:")
print(f"  Number of components: {gmm_iris.n_components}")  # Number of components (3)
print(f"  Silhouette Score: {sil_score_iris:.3f}")  # Quality metric
print(f"  AIC: {gmm_iris.aic(X_iris_scaled):.1f}")  # AIC score (lower is better)
print(f"  BIC: {gmm_iris.bic(X_iris_scaled):.1f}")  # BIC score (lower is better)

# ============================================
# VISUALIZING CLUSTERING RESULTS
# ============================================

# Visualize using first 2 features (for 2D plot)
plt.figure(figsize=(12, 5))  # Figure size: 12×5 inches

# Subplot 1: True labels (ground truth)
plt.subplot(1, 2, 1)  # 1 row, 2 columns, position 1 (left)

# Scatter plot colored by true labels
plt.scatter(X_iris[:, 0], X_iris[:, 1], c=y_iris, cmap='viridis', s=50, alpha=0.7)
# X_iris[:, 0]: First feature (sepal length)
# X_iris[:, 1]: Second feature (sepal width)
# c=y_iris: Color by true species labels

plt.xlabel(iris.feature_names[0])  # X-axis: first feature name
plt.ylabel(iris.feature_names[1])  # Y-axis: second feature name
plt.title('True Labels')  # Chart title
plt.grid(True, alpha=0.3)  # Add grid

# Subplot 2: GMM discovered clusters
plt.subplot(1, 2, 2)  # 1 row, 2 columns, position 2 (right)

# Scatter plot colored by GMM cluster assignments
plt.scatter(X_iris[:, 0], X_iris[:, 1], c=y_iris_pred, cmap='viridis', s=50, alpha=0.7)
# c=y_iris_pred: Color by predicted cluster

plt.xlabel(iris.feature_names[0])  # X-axis: first feature name
plt.ylabel(iris.feature_names[1])  # Y-axis: second feature name
plt.title('GMM Clustering (k=3)')  # Chart title
plt.grid(True, alpha=0.3)  # Add grid

# Adjust layout
plt.tight_layout()
plt.show()  # Display both plots

# ============================================
# EXAMINING PROBABILITY DISTRIBUTIONS
# ============================================

# GMM provides probability distributions - let's examine one sample
sample_idx = 0  # First sample in dataset

print(f"\nSample {sample_idx} probability distribution:")
print(f"  True label: {iris.target_names[y_iris[sample_idx]]}")  # True species name
print(f"  Predicted: {y_iris_pred[sample_idx]}")  # Predicted cluster (hard assignment)

# Display probability distribution
print(f"  Probabilities: {y_iris_proba[sample_idx]}")
# Shows probability of belonging to each cluster

# Display probabilities for each cluster
for i, prob in enumerate(y_iris_proba[sample_idx]):
    print(f"    Cluster {i}: {prob:.3f}")
    # Example: Cluster 0: 0.950, Cluster 1: 0.045, Cluster 2: 0.005
    # This means 95% chance cluster 0, 5% chance others

# Interpretation:
# - High probability (e.g., 0.95) = confident assignment (point is clearly in this cluster)
# - Low probabilities (e.g., 0.05, 0.00) = uncertain assignment (point is near cluster boundary)
# - Probabilities sum to 1.0 (must belong to one of the clusters)
# - This uncertainty information is unique to GMM (K-Means doesn't provide this!)


## Summary & Key Takeaways

### Key Concepts Learned

1. **GMM Basics**
   - Probabilistic clustering model
   - Models data as mixture of Gaussian distributions
   - Provides soft clustering (probability assignments)
   - Uses EM algorithm for parameter estimation

2. **Expectation-Maximization (EM)**
   - **E-Step**: Calculate responsibilities (posterior probabilities)
   - **M-Step**: Update parameters using weighted MLE
   - Iterates until convergence

3. **Model Selection**
   - **AIC (Akaike Information Criterion)**: Penalizes complexity less
   - **BIC (Bayesian Information Criterion)**: Stronger penalty for complexity
   - Lower is better for both
   - Use to select optimal number of components

4. **Covariance Types**
   - **Full**: Each component has own general covariance (most flexible)
   - **Tied**: All components share same covariance
   - **Diag**: Diagonal covariance matrices
   - **Spherical**: Single variance per component (like K-Means)

### When to Use Gaussian Mixture Models

✅ **Good for:**
- Need soft/probabilistic clustering
- Overlapping clusters
- Elliptical clusters (not just spherical)
- When you need probability estimates
- Density estimation
- Continuous data

❌ **Not ideal for:**
- Non-Gaussian data distributions
- Very large datasets (can be slow)
- When hard clustering is sufficient (use K-Means)
- High-dimensional data (curse of dimensionality)
- Discrete/categorical data

### Next Steps

- Compare with **K-Means** for hard clustering
- Use **Bayesian GMM** for automatic component selection
- Apply to **anomaly detection** (low probability points)
- Use for **density estimation** tasks
- Explore **variational inference** for faster training
